# Task 3 — Define Input Schema and Validate- Hanin

## Saudi FinHub — Economic Indicators

This notebook defines and validates the input schema for the cleaned economic indicator datasets:

- GDP
- Inflation
- Unemployment

Rows that pass validation will be kept as valid records, while rows that fail validation will be rejected with the failure reason recorded.

## GDP Schema Definition- Hanin

The following schema defines the expected structure and validation rules for the cleaned GDP dataset.

| Column | Data Type | Nullable | Allowed Values / Range |
|---|---|---|---|
| indicator_name_engl | string | No | Gross Domestic Product-Current prices |
| main_isic_code | string | No | GDP |
| year | integer | No | 2013 or later |
| quarter | string | No | Q1, Q2, Q3, Q4 |
| year_quarter | string | No | YYYY-Q1 to YYYY-Q4 |
| gdp_value | float | No | >= 0 |

## Inflation Schema Definition - Hanin

The following schema defines the expected structure and validation rules for the cleaned Inflation dataset.

| Column | Data Type | Nullable | Allowed Values / Range |
|---|---|---|---|
| item_name | string | No | General Index |
| date | date | No | 2014-01-01 or later |
| inflation_rate | float | No | Numeric value |
| year | integer | No | 2014 or later |
| quarter | string | No | Q1, Q2, Q3, Q4 |
| year_quarter | string | No | YYYY-Q1 to YYYY-Q4 |

## Unemployment Schema Definition - Hnain

The following schema defines the expected structure and validation rules for the cleaned Unemployment dataset.

| Column | Data Type | Nullable | Allowed Values / Range |
|---|---|---|---|
| indicator_name | string | No | Unemployment rate by nationality and sex |
| unemployment_rate | float | No | 0 to 100 |
| is_estimated | boolean | No | True, False |
| year | integer | No | 2013 or later |
| quarter | string | No | Q1, Q2, Q3, Q4 |
| year_quarter | string | No | YYYY-Q1 to YYYY-Q4 |

## Load Cleaned Datasets- Hanin

Load the cleaned economic indicator datasets from `data/interim/` for schema validation.

In [1]:
import pandas as pd
from pathlib import Path

# Define the interim data folder
INTERIM_DATA_PATH = Path("../data/interim")

# Load the cleaned datasets
gdp = pd.read_csv(INTERIM_DATA_PATH / "gdp_cleaned.csv")

inflation = pd.read_csv(
    INTERIM_DATA_PATH / "inflation_cleaned.csv",
    parse_dates=["date"],
    dayfirst=True
)

unemployment = pd.read_csv(
    INTERIM_DATA_PATH / "unemployment_cleaned.csv"
)

# Check that the files loaded correctly
print("GDP shape:", gdp.shape)
print("Inflation shape:", inflation.shape)
print("Unemployment shape:", unemployment.shape)

GDP shape: (53, 5)
Inflation shape: (151, 6)
Unemployment shape: (53, 6)


## GDP Validation - Hanin

Validate the cleaned GDP dataset against the defined schema.

In [2]:
# Create a copy for GDP validation
gdp_validated = gdp.copy()

# Start with all rows marked as valid
gdp_validated["is_valid"] = True
gdp_validated["rejection_reason"] = ""

# Validate required columns for missing values
required_columns = [
    "main_activities",
    "year",
    "quarter",
    "year_quarter",
    "gdp_value"
]

for column in required_columns:
    mask = gdp_validated[column].isnull()
    gdp_validated.loc[mask, "is_valid"] = False
    gdp_validated.loc[mask, "rejection_reason"] += f"Missing {column}; "

# Validate the GDP activity name
mask = gdp_validated["main_activities"] != "Gross Domestic Product"
gdp_validated.loc[mask, "is_valid"] = False
gdp_validated.loc[mask, "rejection_reason"] += "Invalid GDP activity; "

# Validate year range
mask = (gdp_validated["year"] < 2013) | (gdp_validated["year"] > 2026)
gdp_validated.loc[mask, "is_valid"] = False
gdp_validated.loc[mask, "rejection_reason"] += "Year outside expected range; "

# Validate quarter
mask = ~gdp_validated["quarter"].isin(["Q1", "Q2", "Q3", "Q4"])
gdp_validated.loc[mask, "is_valid"] = False
gdp_validated.loc[mask, "rejection_reason"] += "Invalid quarter; "

# Validate GDP value
mask = gdp_validated["gdp_value"] < 0
gdp_validated.loc[mask, "is_valid"] = False
gdp_validated.loc[mask, "rejection_reason"] += "Negative GDP value; "

print("Valid GDP rows:", gdp_validated["is_valid"].sum())
print("Rejected GDP rows:", (~gdp_validated["is_valid"]).sum())

Valid GDP rows: 53
Rejected GDP rows: 0


## Inflation Validation - Hanin

Validate the cleaned Inflation dataset against the defined schema.

In [3]:
# Create a copy for Inflation validation
inflation_validated = inflation.copy()

# Start with all rows marked as valid
inflation_validated["is_valid"] = True
inflation_validated["rejection_reason"] = ""

# Validate required columns for missing values
required_columns = [
    "item_name",
    "date",
    "inflation_rate",
    "year",
    "quarter",
    "year_quarter"
]

for column in required_columns:
    mask = inflation_validated[column].isnull()
    inflation_validated.loc[mask, "is_valid"] = False
    inflation_validated.loc[mask, "rejection_reason"] += f"Missing {column}; "

# Validate item name
mask = inflation_validated["item_name"] != "General Index"
inflation_validated.loc[mask, "is_valid"] = False
inflation_validated.loc[mask, "rejection_reason"] += "Invalid item name; "

# Validate date
mask = inflation_validated["date"] < pd.Timestamp("2014-01-01")
inflation_validated.loc[mask, "is_valid"] = False
inflation_validated.loc[mask, "rejection_reason"] += "Date before 2014; "

# Validate year
mask = inflation_validated["year"] < 2014
inflation_validated.loc[mask, "is_valid"] = False
inflation_validated.loc[mask, "rejection_reason"] += "Year before 2014; "

# Validate quarter
mask = ~inflation_validated["quarter"].isin(["Q1", "Q2", "Q3", "Q4"])
inflation_validated.loc[mask, "is_valid"] = False
inflation_validated.loc[mask, "rejection_reason"] += "Invalid quarter; "

# Validate inflation rate is numeric
mask = pd.to_numeric(
    inflation_validated["inflation_rate"],
    errors="coerce"
).isnull()

inflation_validated.loc[mask, "is_valid"] = False
inflation_validated.loc[mask, "rejection_reason"] += "Invalid inflation rate; "

print("Valid Inflation rows:", inflation_validated["is_valid"].sum())
print("Rejected Inflation rows:", (~inflation_validated["is_valid"]).sum())

Valid Inflation rows: 151
Rejected Inflation rows: 0


## Unemployment Validation- Hanin

Validate the cleaned Unemployment dataset against the defined schema.

In [4]:
# Create a copy for Unemployment validation
unemployment_validated = unemployment.copy()

# Start with all rows marked as valid
unemployment_validated["is_valid"] = True
unemployment_validated["rejection_reason"] = ""

# Validate required columns for missing values
required_columns = [
    "indicator_name",
    "unemployment_rate",
    "is_estimated",
    "year",
    "quarter",
    "year_quarter"
]

for column in required_columns:
    mask = unemployment_validated[column].isnull()
    unemployment_validated.loc[mask, "is_valid"] = False
    unemployment_validated.loc[mask, "rejection_reason"] += f"Missing {column}; "

# Validate indicator name
mask = (
    unemployment_validated["indicator_name"]
    != "Unemployment rate by nationality and sex"
)
unemployment_validated.loc[mask, "is_valid"] = False
unemployment_validated.loc[mask, "rejection_reason"] += "Invalid indicator name; "

# Validate unemployment rate
mask = ~unemployment_validated["unemployment_rate"].between(0, 100)
unemployment_validated.loc[mask, "is_valid"] = False
unemployment_validated.loc[mask, "rejection_reason"] += "Unemployment rate outside 0-100; "

# Validate is_estimated
mask = ~unemployment_validated["is_estimated"].isin([True, False])
unemployment_validated.loc[mask, "is_valid"] = False
unemployment_validated.loc[mask, "rejection_reason"] += "Invalid is_estimated value; "

# Validate year
mask = unemployment_validated["year"] < 2013
unemployment_validated.loc[mask, "is_valid"] = False
unemployment_validated.loc[mask, "rejection_reason"] += "Year before 2013; "

# Validate quarter
mask = ~unemployment_validated["quarter"].isin(["Q1", "Q2", "Q3", "Q4"])
unemployment_validated.loc[mask, "is_valid"] = False
unemployment_validated.loc[mask, "rejection_reason"] += "Invalid quarter; "

print("Valid Unemployment rows:", unemployment_validated["is_valid"].sum())
print("Rejected Unemployment rows:", (~unemployment_validated["is_valid"]).sum())

Valid Unemployment rows: 53
Rejected Unemployment rows: 0


## Separate Valid and Rejected Records - Hanin

Separate records that passed validation from rejected records and retain the rejection reason for failed rows.

In [5]:
# Separate valid and rejected GDP rows
gdp_valid = gdp_validated[gdp_validated["is_valid"]].copy()
gdp_rejected = gdp_validated[~gdp_validated["is_valid"]].copy()

# Separate valid and rejected Inflation rows
inflation_valid = inflation_validated[
    inflation_validated["is_valid"]
].copy()

inflation_rejected = inflation_validated[
    ~inflation_validated["is_valid"]
].copy()

# Separate valid and rejected Unemployment rows
unemployment_valid = unemployment_validated[
    unemployment_validated["is_valid"]
].copy()

unemployment_rejected = unemployment_validated[
    ~unemployment_validated["is_valid"]
].copy()

# Display validation results
print("GDP:", len(gdp_valid), "valid,", len(gdp_rejected), "rejected")
print("Inflation:", len(inflation_valid), "valid,", len(inflation_rejected), "rejected")
print("Unemployment:", len(unemployment_valid), "valid,", len(unemployment_rejected), "rejected")

GDP: 53 valid, 0 rejected
Inflation: 151 valid, 0 rejected
Unemployment: 53 valid, 0 rejected


## Save Validation Outputs - Hanin

Save the validated and rejected records for each economic indicator dataset.

In [6]:
# Remove validation helper columns from valid records
gdp_valid = gdp_valid.drop(
    columns=["is_valid", "rejection_reason"]
)

inflation_valid = inflation_valid.drop(
    columns=["is_valid", "rejection_reason"]
)

unemployment_valid = unemployment_valid.drop(
    columns=["is_valid", "rejection_reason"]
)

# Save validated datasets
gdp_valid.to_csv(
    INTERIM_DATA_PATH / "gdp_validated.csv",
    index=False
)

inflation_valid.to_csv(
    INTERIM_DATA_PATH / "inflation_validated.csv",
    index=False,
    date_format="%d-%m-%Y"
)

unemployment_valid.to_csv(
    INTERIM_DATA_PATH / "unemployment_validated.csv",
    index=False
)

# Save rejected datasets
gdp_rejected.to_csv(
    INTERIM_DATA_PATH / "gdp_rejected.csv",
    index=False
)

inflation_rejected.to_csv(
    INTERIM_DATA_PATH / "inflation_rejected.csv",
    index=False,
    date_format="%d-%m-%Y"
)

unemployment_rejected.to_csv(
    INTERIM_DATA_PATH / "unemployment_rejected.csv",
    index=False
)

print("Validation files saved successfully.")

Validation files saved successfully.
